<a href="https://colab.research.google.com/github/PauDK/Deceptive-Review-Detection/blob/main/BERT_Transfer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow-text
!pip install tf-models-official

In [ ]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_hub as hub
import tensorflow_text as text
from official.nlp import optimization

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from pprint import pprint

In [ ]:
print(tf.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd '/content/drive/MyDrive/Github/Deceptive-Review-Detection'

In [ ]:
df = pd.read_csv("Chicago_Hotel_Review/Chicago_Hotel_Reviews.csv", index_col=0)
df = df.drop('Rating', axis=1)
df.loc[df['Label'] == -1, 'Label'] = 0
X = df['Ori_Review']
y = df['Label']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [ ]:
preprocessor = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")

In [ ]:
text_inputs = [keras.layers.Input(shape=(), dtype=tf.string)]
tokenize = hub.KerasLayer(preprocessor.tokenize)
tokenized_inputs = [tokenize(segment) for segment in text_inputs]

seq_length = 512  # Set sequence length
bert_pack_inputs = hub.KerasLayer(
    preprocessor.bert_pack_inputs,
    arguments=dict(seq_length=seq_length))
preprocessed_text = bert_pack_inputs(tokenized_inputs)

In [ ]:
# sizes = []
# tokens = tokenize(X_train)
# for i in range(tokens.shape[0]):
#   sizes.append(tokens[i].shape[0])

# from matplotlib import pyplot as plt

# fig, (ax1, ax2) = plt.subplots(1, 2)
# fig.set_figwidth(12)
# ax1.hist(sizes)
# ax2.boxplot(sizes)

In [ ]:
text_inputs

In [ ]:
preprocessed_text['input_mask']

In [ ]:
bert_encoder = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/4", trainable=True)

In [ ]:
# Train all BERT layers as suggested by BERT developers
bert_encoder.trainable = True

# Bert layers
outputs = bert_encoder(preprocessed_text)

# Dense layers
l = keras.layers.Dense(1, activation='sigmoid', name="output") (outputs['pooled_output'])

# Use inputs and outputs to construct a final model
model = keras.Model(inputs=text_inputs, outputs = [l])

In [ ]:
epochs = 3
batch_size = 8
init_lr = 3e-5  # best of [5e-5, 3e-5, 2e-5]

train_data_size = X_train.shape[0]
steps_per_epoch = train_data_size // batch_size
num_train_steps = steps_per_epoch * epochs
num_warmup_steps = num_train_steps // 10

test_data_size = X_test.shape[0]

In [ ]:
optimizer = optimization.create_optimizer(
      init_lr=init_lr,
      num_train_steps=num_train_steps,
      num_warmup_steps=num_warmup_steps,
      optimizer_type='adamw')

# optimizer = optimization.create_optimizer(
#       init_lr=init_lr,
#       num_train_steps=num_train_steps,
#       num_warmup_steps=0,
#       optimizer_type='adamw')

model.compile(optimizer=optimizer,
              loss='binary_crossentropy',
              metrics='accuracy')

model.summary()

In [ ]:
bert_encoder.trainable = False
model.fit(X_train,
          y_train,
          steps_per_epoch=steps_per_epoch,
          epochs=epochs,
          validation_data=(X_test, y_test))

In [ ]:
prediction = model.predict(X_test)
prediction_real = model.predict(X_test[y_test == 0])
prediction_fake = model.predict(X_test[y_test == 1])

In [ ]:
prediction_bi_real = prediction_real.copy()
prediction_bi_real[np.where(prediction_real <= 0.5)] = 0
prediction_bi_real[np.where(prediction_real > 0.5)] = 1
print('True Positive Rate:', '{:.2%}'.format(1 - sum(prediction_bi_real)[0] / len(prediction_bi_real)))

prediction_bi_fake = prediction_fake.copy()
prediction_bi_fake[np.where(prediction_fake <= 0.5)] = 0
prediction_bi_fake[np.where(prediction_fake > 0.5)] = 1
print('False Positive Rate:', '{:.2%}'.format(sum(prediction_bi_fake)[0] / len(prediction_bi_fake)))

In [ ]:
demo_ind = np.random.choice(X_test.index)
realfake = 'Real' if y_test[demo_ind] == 0 else 'Fake'
prediction = model.predict([X_test[demo_ind]])
prediction_rf = 'Real' if prediction <= 0.5 else 'Fake'

print(f'{realfake} review to be fed to model:\n', X_test[demo_ind], '\n')
print(f'Model prediction: {prediction_rf}')

In [ ]:
import os

main_save_path = '/content/drive/My Drive/Colab Notebooks/my_models'
saved_model_name = 'Fine_Tune_Bert_Accu_92-19'
saved_model_path = os.path.join(main_save_path, saved_model_name)
save_options = tf.saved_model.SaveOptions(experimental_io_device='/job:localhost')

model.save(saved_model_path, include_optimizer=False, options=save_options)